# Challenge One: Real-Time Weather Alerts Agent

```
Nathan Verrill
student_04_a8b20623816c_Nathan_Verrill
June 2026
```

This notebook implements a standalone weather application using custom tool function grounding combined with the Google Agent Development Kit (ADK) framework.

### Cell 1: Environment Setup & Package Installations

In [ ]:
import os
import getpass

# Install the core Google Agent Development Kit framework package
!pip install -q google-adk requests

# Configure baseline target environment parameters
os.environ["GOOGLE_CLOUD_PROJECT"] = "local-agent-development"

# Securely prompt and set the API Key environment variable
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Cloud / AI Studio API Key: ")

print("\n✅ Environment successfully configured.")

### Cell 2: Define Custom Weather and Google Maps Geocoding Tools

In [ ]:
import os
import requests
from typing import Dict, Any

def get_nws_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    """Retrieves the current weather forecast from the National Weather Service API.

    Args:
        latitude (float): Latitude of the location (e.g., 40.7128).
        longitude (float): Longitude of the location (e.g., -74.0060).

    Returns:
        Dict[str, Any]: A dictionary containing the weather periods or forecast metrics.
    """
    headers = {"User-Agent": "LocalWeatherAgent/1.0 (agent-development@local.internal)"}
    try:
        points_url = f"https://api.weather.gov/points/{latitude:.4f},{longitude:.4f}"
        res = requests.get(points_url, headers=headers)
        if res.status_code != 200:
            return {"error": f"NWS Points API returned status code {res.status_code}"}
        
        forecast_url = res.json()["properties"]["forecast"]
        forecast_res = requests.get(forecast_url, headers=headers)
        if forecast_res.status_code != 200:
            return {"error": f"NWS Forecast API returned status code {forecast_res.status_code}"}
            
        return {"forecast": forecast_res.json()["properties"]["periods"][0]}
    except Exception as e:
        return {"error": str(e)}


def geocode_place(address: str) -> Dict[str, Any]:
    """Converts a descriptive city name or address into latitude and longitude coordinates.

    Args:
        address (str): The descriptive address or city name (e.g., "Austin, TX").

    Returns:
        Dict[str, Any]: A dictionary containing 'lat' and 'lng' float values.
    """
    api_key = os.getenv("GOOGLE_API_KEY")
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={api_key}"
    try:
        res = requests.get(url)
        data = res.json()
        if data.get("status") == "OK":
            location = data["results"][0]["geometry"]["location"]
            return {"lat": float(location["lat"]), "lng": float(location["lng"])}
        return {"error": f"Google Maps Geocoding failed with status: {data.get('status')}"}
    except Exception as e:
        return {"error": str(e)}

print("✅ Custom tool functions compiled in memory.")

### Cell 3: Instantiating the Independent Grounded Weather Agent

In [ ]:
from google.adk.agents import Agent

WEATHER_AGENT_INSTRUCTIONS = """
You are a helpful Weather Assistant Agent.
When a user asks for weather or conditions in a specific location, city, or address:
1. First use 'geocode_place' to resolve the descriptive name into geographic latitude and longitude coordinates.
2. Next, use 'get_nws_weather' with those explicit coordinates to fetch real-time forecasts.
3. Combine and parse this context to provide a clear summary back to the user.
"""

# Construct the agent attaching custom function tools
weather_agent = Agent(
    name="Pat",
    model="gemini-2.5-flash",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[geocode_place, get_nws_weather]
)

print("✅ Real-Time Weather Alerts Agent successfully initialized.")

### Cell 4: Application Verification Loop across Target Locations

In [ ]:
from vertexai.preview import reasoning_engines

# Host the agent within the application orchestration environment wrapper
app = reasoning_engines.AdkApp(agent=weather_agent)

# Local validation verification scenario prompts targeting multiple US cities
test_locations = [
    "What is the weather like in Austin, TX?",
    "Can you check the current conditions in St. Louis, MO?",
    "What is the forecast right now in Boston, MA?"
]

print("🚀 Querying weather application host natively across test parameters...\n")
user_id = "local-challenge-one-session"

for query in test_locations:
    print(f"=======================================================")
    print(f"💬 Prompt: '{query}'")
    print(f"=======================================================")
    
    try:
        # Generate an independent tracking state session
        session = app.create_session(user_id=user_id)
        session_id = session.get("session_id") or session.get("id")
        
        # Stream execution query context chunks
        response_text = ""
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=query
        ):
            if isinstance(event, dict) and "content" in event and "parts" in event["content"]:
                for part in event["content"]["parts"]:
                    if "text" in part:
                        response_text += part["text"]
                        
        if response_text:
            print(f"✨ [Agent Answer Output]:\n{response_text.strip()}\n")
            
    except Exception as e:
        print(f"⚠️ Pipeline Execution Error: {e}\n")

print("🏁 Notebook validation verification loop complete.")